# MNAI Imagine 1 — 250M from scratch — Kaggle, step by step

Same training as the Colab notebook, but it runs on Kaggle. After you commit it with Save & Run All, it trains on Kaggle's servers with your laptop closed. One run stops cleanly after 10.5 hours with a checkpoint saved, inside Kaggle's 12-hour cap. Resume is automatic: if Step 5 restores a checkpoint, Step 6 picks it up on its own.

Moving from Colab: when your Colab session ends, download the newest ckpt folder from Drive, upload it as a Kaggle dataset, attach it as an input, and Step 5 brings it back.

## Step 1 — Check the GPU

In [ ]:
!nvidia-smi

## Step 2 — Clone the repo

In [ ]:
!git clone https://github.com/MeowSamOPDZN/mnai-imagine-1.git
%cd mnai-imagine-1
!ls -lh

## Step 3 — Install everything

In [ ]:
!pip install -q -r requirements.txt

## Step 4 — Check the model shape

In [ ]:
!python model.py

## Step 5 — Bring back last night's checkpoint (skip on the first night)

In [ ]:
!mkdir -p /kaggle/working/out-im1
import glob, os, shutil
found = sorted(glob.glob('/kaggle/input/**/ckpt-*', recursive=True))
for ck in found:
    name = os.path.basename(ck)
    dest = f'/kaggle/working/out-im1/{name}'
    if not os.path.exists(dest):
        print('restoring', ck, '->', dest)
        shutil.copytree(ck, dest)
print('checkpoints in /kaggle/working/out-im1:', sorted(os.listdir('/kaggle/working/out-im1')))

## Step 6 — Train the model (starts fresh, or resumes on its own)

One cell does both jobs. If Step 5 restored a checkpoint, it resumes from the newest one. If there is nothing to restore, it starts from scratch. You never have to change a checkpoint number.

In [ ]:
import glob, os, subprocess
ckpts = sorted(glob.glob('/kaggle/working/out-im1/ckpt-*'), key=lambda p: int(p.rsplit('-', 1)[-1]))
cmd = ['python', 'train.py',
       '--out', '/kaggle/working/out-im1', '--max-images', '1000000',
       '--micro-batch', '16', '--grad-accum', '4', '--max-hours', '10.5']
if ckpts:
    print('resuming from', ckpts[-1])
    cmd += ['--resume', ckpts[-1]]
else:
    print('no checkpoint found, starting fresh')
subprocess.run(cmd, check=True)

## Step 7 — See what saved so far

In [ ]:
!ls /kaggle/working/out-im1

## Step 8 — Make pictures

In [ ]:
import glob, subprocess
ckpts = sorted(glob.glob('/kaggle/working/out-im1/ckpt-*'), key=lambda p: int(p.rsplit('-', 1)[-1]))
model = ckpts[-1] if ckpts else '/kaggle/working/out-im1/ckpt-500'
print('generating from', model)
subprocess.run(['python', 'generate.py', '--model', model,
                '--prompt', 'a photo of a cat', '--n', '4', '--out', 'sample.png'], check=True)

## Step 9 — Try the other prompts

In [ ]:
import glob, subprocess
ckpts = sorted(glob.glob('/kaggle/working/out-im1/ckpt-*'), key=lambda p: int(p.rsplit('-', 1)[-1]))
model = ckpts[-1] if ckpts else '/kaggle/working/out-im1/ckpt-500'
print('generating from', model)
subprocess.run(['python', 'generate.py', '--model', model,
                '--prompt', 'a photo of a dog', '--n', '4', '--out', 'sample-dog.png'], check=True)

## Step 10 — Download your pictures

In [ ]:
!ls -lh sample*.png

## If something goes wrong

- **No GPU in Step 1:** Settings on the right > Accelerator > GPU T4 x2, then Save Version again. If Kaggle says no GPU is available, wait a bit and try again later.
- **Clone failed, could not resolve host:** Internet is off. Settings on the right > Internet ON, then re-run the cell.
- **Out of memory in Step 6:** lower micro-batch to 8: change --micro-batch 16 --grad-accum 4 to --micro-batch 8 --grad-accum 8.
- **Version did not save:** the run went past 12 hours. Lower --max-hours (try 10) and commit again.
- **Session ended:** start a new version, attach the last version's output as an input, run Step 5 to restore the checkpoint, then Step 6 — it resumes on its own.
- **First night:** Step 5 finds nothing, that is fine. Step 6 starts fresh.
- **Moving from Colab:** download the newest ckpt folder from Drive, upload it as a Kaggle dataset, attach it as an input, then Step 5 restores it and Step 6 resumes.